# Case 900 VDI transfer validation

This notebook reports the predeclared F36-like/F52-like transfer test. It does not perform a Case 900 factorial search. Existing results are loaded by default; set `RUN_SIMULATIONS=True` only to regenerate this Case 900 result directory.

In [1]:
from pathlib import Path
import sys, numpy as np, pandas as pd
HERE=Path.cwd().resolve()
if HERE.name!='2_vdi': HERE=HERE/'2_validation/_BESTEST/2_vdi'
sys.path.insert(0,str(HERE/'doc'))
import bestest_vdi_transfer_engine as engine
RUN_SIMULATIONS=False
OUT=HERE/'results/case900_vdi_transfer'

## Case 600 transfer-engine equivalence gate

The generalized runner must reproduce the stored F36/F52 result before Case 900 is regenerated. The completed implementation check produced exactly zero annual-load difference. It is not rerun when `RUN_SIMULATIONS=False`.

In [2]:
if RUN_SIMULATIONS:
    replay=engine.verify_case600_replay(tolerance=1e-9)
    summary,hourly,layers=engine.run_case900(OUT)
else:
    summary=pd.read_csv(OUT/'case900_vdi_transfer_summary.csv')
    hourly=pd.read_csv(OUT/'case900_vdi_transfer_hourly.csv')
    layers=pd.read_csv(OUT/'case900_layer_capacity_audit.csv')
assert len(summary)==2 and len(hourly)==2*8760
assert np.isfinite(summary.select_dtypes('number')).all().all()
assert summary.maximum_balance_residual_W.max()<=1e-9

## Input translation audit

In [3]:
audit=pd.read_csv(OUT/'case900_input_translation_audit.csv')
checklist=pd.read_csv(OUT/'case900_eight_point_checklist.csv')
assert np.isclose(layers.total_C_J_K.sum(),14_772_000.0,atol=1e-6)
display(audit)
display(layers.round(6))
display(checklist)

,item,current_setting,classification,source_provenance
0,geometry,48 m2; 129.6 m3; same surfaces,unchanged from Case 600,inputs/case900.json
1,construction layers,Case600 identities; cp x 5.405964196971026,comparator-derived,capacity-constrained proxy
2,prescribed U-values,wall .53; roof .33; floor .038 W/m2K,unchanged from Case 600,inputs/case900.json
3,thermal mass/capacitance,heavyweight; 14.772 MJ/K,benchmark-prescribed change,inputs/case900.json
4,effective mass area,"2.43 x 48 = 116.64 m2; recorded, not applied",unresolved provenance,no direct no-IW VDI mapping
5,glazing,12 m2 south; U 3.1; g .769,unchanged from Case 600,inputs/case900.json
6,infiltration/ventilation,0.414 1/h; no mechanical ventilation,unchanged from Case 600,repository comparator; canonical provenance open
7,internal gains,200 W; 50% air/50% AW,implementation assumption,comparator-consistent no-IW projection
8,solar properties,native VDI; absorptance .6; g .769,unchanged from Case 600,candidate implementation
9,setpoints/control,20/27 C; ideal air HVAC,unchanged from Case 600,benchmark/comparator


,assembly,area_m2,layer_R_m2K_W,layer_C_J_m2K,total_C_J_K
0,wall,63.6,1.789286,78571.797309,4.997166e+06
1,roof,48.0,2.993214,98226.066725,4.714851e+06
2,floor,48.0,25.253571,105416.301841,5.059982e+06


,case,category,verification_question,current_setting,source_provenance
0,900,IW topology,Is the intended topology present and are absen...,no-IW,Case600 candidate
1,900,Solar/source allocation,Which nodes receive solar and is allocation na...,F36/F52 variants,Case600 candidate
2,900,Envelope convention,Which inputs control transmission and dynamics...,layer-controlled capacity proxy,Case900 aggregate mass + implementation proxy
3,900,Inside heat-transfer treatment,Which total convention is active and how is it...,total h_i=8 W/m2K,VDI-side convention
4,900,Exterior long-wave treatment,Is long-wave native or harmonised and what is ...,harmonised,Case600 candidate
5,900,Internal-gain allocation,Which nodes receive gains and what is the sour...,50% air / 50% AW,comparator projection
6,900,Airflow/infiltration provenance,"What airflow quantity and provenance are used,...",0.414 1/h,repository comparator; canonical source open
7,900,"Geometry, schedules and controls","Do geometry, schedules, setpoints, availabilit...",benchmark geometry/schedules/controls,canonical case record


## Annual heating/cooling and BESTEST checks

Distances are measured to the accepted intervals, not their centres.

In [4]:
cols=['case','configuration','heating_MWh','cooling_MWh','heating_lower','heating_upper','cooling_lower','cooling_upper','heating_in_range','cooling_in_range','both_in_range','D_MWh_norm','maximum_balance_residual_W']
display(summary[cols].round(6))

,case,configuration,heating_MWh,cooling_MWh,heating_lower,heating_upper,cooling_lower,cooling_upper,heating_in_range,cooling_in_range,both_in_range,D_MWh_norm,maximum_balance_residual_W
0,900,F36-like,2.521021,2.783916,1.04,2.28,2.35,2.6,False,False,False,0.760908,0.0
1,900,F52-like,2.509007,2.839874,1.04,2.28,2.35,2.6,False,False,False,0.977110,0.0


## Stop/go decision

Both candidates materially fail because `D_MWh_norm > 0.25`. The wider 6XX/9XX simulations are therefore stopped. The next permissible work is a mass-relevant diagnosis of the capacity-constrained heavyweight proxy and its placement in the no-IW VDI network—not a new factorial optimization. The checked-in Case 900 cooling range also remains provenance-sensitive because it excludes the repository Modelica result.